# 02 · Extrair um perfil estruturado

In [1]:
import os, sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))
from pprint import pprint
from src.config import DATA

<a id="arquitetura"></a>

**Anatomia.** O modelo é escolhido via `OPENAI_MODEL`. O modo sem chave usa um adaptador
determinístico. A mensagem de sistema limita a extração a fatos do currículo. As ferramentas
previstas são busca, câmbio, SQL e FAISS; o currículo e os anúncios formam a base de conhecimento.

**Padrão e maturidade.** A chain MVP é fixa. O integrador será híbrido: reage a pedidos
de busca e aplica uma sequência deliberada para comparar vagas ou preparar uma candidatura.

**Planejamento.** Encontrar vagas → extrair perfil → interpretar restrições → consultar fontes
→ filtrar → ranquear → verificar. Memória curta mantém turnos recentes e a ordem da lista.
A memória longa representa perfil, preferências e descartes recuperáveis por similaridade.

In [2]:
from src.config import live_model
from src.prompts import perfil_chain, resumo_chain
from src.demo import perfil_imperativo
cv = (DATA / 'curriculo.txt').read_text()
llm = live_model() if os.getenv('NOTEBOOK_OPENAI') == '1' else None
chain = perfil_chain(llm)
print('invoke:'); pprint(chain.invoke({'curriculo': cv}).model_dump())
print('batch:')
for result in chain.batch([{'curriculo': text} for text in (
    cv, 'Skills: Java, Docker\nInglês: B1', 'Skills: TypeScript, React')]):
    pprint(result.model_dump())
print('stream:', end=' ')
for fragment in resumo_chain(llm).stream({'vaga': 'Worldwide remote. Python and Docker. Salary not disclosed.'}):
    print(fragment, end='')
print()
print('@chain:', perfil_imperativo.invoke(cv).model_dump())

invoke:


{'cargos_alvo': ['Senior Backend Engineer'],
 'ingles': 'B2',
 'pretensao_salarial': None,
 'senioridade': 'Sênior',
 'skills': ['Python', 'FastAPI', 'PostgreSQL', 'Docker']}
batch:


{'cargos_alvo': ['Senior Backend Engineer'],
 'ingles': 'B2',
 'pretensao_salarial': None,
 'senioridade': 'sênior',
 'skills': ['Python', 'FastAPI', 'PostgreSQL', 'Docker']}
{'cargos_alvo': [],
 'ingles': 'B1',
 'pretensao_salarial': None,
 'senioridade': None,
 'skills': ['Java', 'Docker']}
{'cargos_alvo': [],
 'ingles': None,
 'pretensao_salarial': None,
 'senioridade': None,
 'skills': ['TypeScript', 'React']}
stream: 

Vaga remota, aberta a candidatos do mundo todo. Requer conhecimentos em Python e

 Docker. Salário não divulgado.
@chain: {'skills': ['Python', 'FastAPI', 'PostgreSQL', 'Docker'], 'senioridade': 'senior', 'cargos_alvo': ['Senior Backend Engineer'], 'ingles': 'B2', 'pretensao_salarial': None}


O encadeamento LCEL isola template, modelo, parser e validação. Isso permite
        trocar um componente sem duplicar a sequência e adicionar retry no ponto da falha.
        A versão imperativa usa uma condição de entrada vazia e reaproveita a composição.

## Evidência

- **1.1** → [arquitetura](#arquitetura): Anatomia, padrão híbrido e maturidade.
- **1.2** → [arquitetura](#arquitetura): Decomposição e memória curta/longa.
- **1.3** → [lcel](#lcel): Templates de chat e texto.
- **1.4** → [lcel](#lcel): Parser JSON seguido de validação Pydantic.
- **1.5** → [lcel](#lcel): Invoke, batch, stream e @chain.
- **1.6** → [lcel](#lcel): Composição LCEL e justificativa.